# Working with Grid Grobs

A **grob** (graphical object) is the fundamental drawable entity in grid_py.
Rather than issuing immediate drawing commands, grid_py encourages you to
create grob objects, inspect and edit them, and then draw them.

This notebook covers:

* Creating grobs with constructor functions (`rect_grob`, `circle_grob`, `text_grob`, etc.).
* Editing grobs with `edit_grob`.
* Grouping grobs into trees with `GTree` and `GList`.
* Navigating grob hierarchies with `GPath`.

The narrative follows the original R vignette (`grobs.Rnw`).

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grid_py as gp

## Creating Grobs

Every primitive drawing function in grid_py has a `*_grob` counterpart
that returns the grob *without* drawing it. This lets you build up
a scene programmatically before rendering.

In [ ]:
# Create individual grobs (not yet drawn)
r = gp.rect_grob(
    width=gp.Unit(0.8, "npc"),
    height=gp.Unit(0.5, "npc"),
    gp=gp.Gpar(fill="lightblue", col="steelblue"),
    name="my_rect",
)

c = gp.circle_grob(
    r=gp.Unit(0.2, "npc"),
    gp=gp.Gpar(fill="salmon", col="darkred"),
    name="my_circle",
)

t = gp.text_grob(
    "grob demo",
    gp=gp.Gpar(fontsize=16),
    name="my_text",
)

print(r)
print(c)
print(t)

In [ ]:
# Draw the grobs one by one
gp.grid_newpage()
gp.grid_draw(r)
gp.grid_draw(c)
gp.grid_draw(t)

## Editing Grobs

The `edit_grob` function returns a *copy* of a grob with certain attributes
changed. The original grob is left unmodified.

In [ ]:
# Edit the rectangle to change its fill colour
r2 = gp.edit_grob(r, gp=gp.Gpar(fill="lightyellow", col="orange"))

gp.grid_newpage()
gp.grid_draw(r2)
gp.grid_draw(t)

In [ ]:
# The original grob is unchanged
print("original fill:", r._gp)
print("edited   fill:", r2._gp)

## GTree -- Grouping Grobs

A `GTree` is a grob that contains child grobs. It is the primary
mechanism for building composite graphical objects. Children are
stored in a `GList`.

In [ ]:
# Build a GTree from individual grobs
children = gp.GList(r, c, t)
tree = gp.GTree(children=children, name="my_tree")

print("GTree:", tree)
print("Number of children:", len(gp.child_names(tree)))
print("Child names:", gp.child_names(tree))

In [ ]:
# Draw the entire tree at once
gp.grid_newpage()
gp.grid_draw(tree)

In [ ]:
# Add a grob to an existing tree
line = gp.lines_grob(
    x=gp.Unit([0.1, 0.9], "npc"),
    y=gp.Unit([0.3, 0.7], "npc"),
    gp=gp.Gpar(col="purple", lwd=3),
    name="my_line",
)
tree2 = gp.add_grob(tree, line)
print("Updated child names:", gp.child_names(tree2))

## GPath Navigation

The `GPath` class lets you address grobs deep inside a hierarchy by
a dotted path such as `"my_tree::my_rect"`. Functions like `get_grob`,
`set_grob`, and `remove_grob` accept path arguments.

In [ ]:
# Retrieve a child by name
child = gp.get_grob(tree, "my_rect")
print("Retrieved child:", child)

In [ ]:
# Replace a child inside the tree
new_rect = gp.rect_grob(
    width=gp.Unit(0.5, "npc"),
    height=gp.Unit(0.3, "npc"),
    gp=gp.Gpar(fill="palegreen", col="darkgreen"),
    name="my_rect",
)
tree3 = gp.set_grob(tree, new_rect)
print("Child names after set:", gp.child_names(tree3))

In [ ]:
# Remove a child
tree4 = gp.remove_grob(tree, "my_circle")
print("After removal:", gp.child_names(tree4))

## Summary

* `*_grob` functions create grob objects without drawing.
* `edit_grob` returns modified copies of grobs.
* `GTree` and `GList` organise grobs into hierarchies.
* `get_grob`, `set_grob`, `add_grob`, and `remove_grob` let you
  manipulate children by name or path.